# Vibe Coding: LLMs, Tools y Agentes

De escribir codigo a mano a evaluar codigo generado por IA.

## Objetivo de la sesion

- entender que patrones usa una IA al generar codigo Python
- aprender `try/except`, type hints, `@property`, comprehensions y decoradores
- saber leer y evaluar codigo que no has escrito tu

Cada celda sigue este patron: **"una IA genero esto → que hace? → pruebalo → ahora entiendes el patron."**

## 0. El puente desde Lesson 2

En Lesson 2 escribiste estas clases a mano: `Order`, `Trade`, `PositionTracker`. Unas ~60 lineas.

Ahora imagina que le pides a un LLM: *"Genera un PositionTracker con type hints y @property"*. Lo que devuelve se parece a lo tuyo — pero usa patrones que aun no conoces.

**Antes de ejecutar:** lee el codigo de abajo. ¿Que lineas te resultan nuevas respecto a tu version de L2?

In [ ]:
# --- Lo que escribiste en L2 ---
class Trade:
    def __init__(self, symbol, side, price, size):
        self.symbol = symbol
        self.side = side
        self.price = price
        self.size = size

    def cash_flow(self):
        signed = -1 if self.side == "buy" else 1
        return signed * self.price * self.size


# --- Lo que un LLM podria generar ---
class PositionTracker:
    """Tracks cash and position for a single asset."""

    def __init__(self) -> None:
        self._cash: float = 0.0
        self._position: float = 0.0

    @property
    def cash(self) -> float:
        """Current cash balance (read-only)."""
        return self._cash

    @property
    def position(self) -> float:
        """Current position size (read-only)."""
        return self._position

    def apply_trade(self, trade: Trade) -> None:
        """Apply a trade to update cash and position."""
        if trade.side == "buy":
            self._cash -= trade.price * trade.size
            self._position += trade.size
        else:
            self._cash += trade.price * trade.size
            self._position -= trade.size

    def equity(self, mark_price: float) -> float:
        """Calculate current equity at given mark price."""
        return self._cash + self._position * mark_price

    def __repr__(self) -> str:
        return f"PositionTracker(cash={self._cash:.2f}, pos={self._position:.4f})"


# Probamos
tracker = PositionTracker()
tracker.apply_trade(Trade("BTCUSDT", "buy", 100000, 0.10))
print("cash:", tracker.cash)         # <- ¿por que no tracker._cash?
print("position:", tracker.position) # <- ¿que hace @property?
print("equity:", tracker.equity(100080))

## 1. `try / except` — cuando el codigo de la IA crashea

Un LLM puede generar codigo que falla en runtime. En vez de que tu programa explote, `try/except` captura el error y te deja reaccionar.

**Antes de ejecutar:** ¿que error esperas en la primera division?

In [ ]:
# Sin try/except: el programa crashea
# price = 100000
# size = 0
# avg_price = price / size  # ZeroDivisionError!

# Con try/except: el programa sobrevive
def safe_avg_price(price: float, size: float) -> float:
    try:
        return price / size
    except ZeroDivisionError:
        print("WARNING: size is zero, returning 0.0")
        return 0.0

print("normal:", safe_avg_price(100000, 0.10))
print("zero:  ", safe_avg_price(100000, 0))
print()

# Patron real: capturar errores al procesar multiples trades
trades_data = [
    {"symbol": "BTCUSDT", "price": 100000, "size": 0.10},
    {"symbol": "ETHUSDT", "price": 0, "size": 1.5},       # precio cero — raro pero posible
    {"symbol": "BTCUSDT", "price": 99980, "size": 0.05},
]

results = []
for t in trades_data:
    try:
        notional = t["price"] * t["size"]
        if t["price"] == 0:
            raise ValueError(f"precio invalido para {t['symbol']}")
        results.append({"symbol": t["symbol"], "notional": notional})
    except ValueError as e:
        print(f"SKIP: {e}")

print("trades procesados:", len(results))
for r in results:
    print(f"  {r['symbol']}: notional={r['notional']}")

## 2. Type hints — leer las firmas que genera la IA

Un LLM casi siempre genera codigo con type hints. No cambian como funciona el programa, pero te dicen **que tipo espera cada parametro y que devuelve la funcion**.

**Antes de ejecutar:** lee las firmas y predice los tipos de retorno.

In [ ]:
# Sin type hints (L2)
def compute_notional_v1(price, size):
    return price * size

# Con type hints (lo que genera una IA)
def compute_notional_v2(price: float, size: float) -> float:
    return price * size

# Ambas hacen EXACTAMENTE lo mismo
print(compute_notional_v1(100000, 0.10))
print(compute_notional_v2(100000, 0.10))

# Pero los type hints son documentacion automatica:
# - price: float  → "espera un numero decimal"
# - size: float   → "espera un numero decimal"
# - -> float      → "devuelve un numero decimal"

# Ejemplo mas complejo: una IA genera esto
def filter_trades(
    trades: list[dict],
    side: str = "buy",
    min_size: float = 0.0,
) -> list[dict]:
    """Filter trades by side and minimum size."""
    return [t for t in trades if t["side"] == side and t["size"] >= min_size]

# Puedes leer la firma sin mirar el cuerpo:
# - trades: lista de diccionarios
# - side: string, por defecto "buy"
# - min_size: float, por defecto 0.0
# - devuelve: lista de diccionarios

all_trades = [
    {"symbol": "BTCUSDT", "side": "buy", "price": 100000, "size": 0.10},
    {"symbol": "BTCUSDT", "side": "sell", "price": 100020, "size": 0.08},
    {"symbol": "ETHUSDT", "side": "buy", "price": 3520, "size": 0.03},
]

buys = filter_trades(all_trades, side="buy", min_size=0.05)
print("big buys:", buys)

## 3. `@property` — acceso controlado al estado interno

En Lesson 2 usabas `tracker._cash` (con guion bajo, convencion de "no tocar"). La IA genera `@property` para dar acceso de lectura sin exponer el atributo interno.

**Antes de ejecutar:** ¿que diferencia esperas entre `tracker.cash` y `tracker._cash`?

In [ ]:
# @property convierte un metodo en algo que parece un atributo
# Arriba ya definimos PositionTracker con @property

tracker = PositionTracker()
tracker.apply_trade(Trade("BTCUSDT", "buy", 100000, 0.10))

# Estas dos lineas devuelven lo mismo:
print("tracker._cash:   ", tracker._cash)     # acceso directo (L2)
print("tracker.cash:    ", tracker.cash)       # via @property (L3)

# Pero hay una diferencia importante:
# tracker._cash = 999  → Python lo permite (pero rompe la logica)
# tracker.cash = 999   → ¿que pasa?

try:
    tracker.cash = 999
except AttributeError as e:
    print(f"\nError al asignar: {e}")
    print("@property sin setter = solo lectura. Nadie puede romper el estado.")

print("\n--- Resumen ---")
print("_cash:     convencion — 'por favor no toques esto'")
print("@property: mecanismo — 'Python te impide tocarlo'")

## 4. List comprehensions — lo que la IA escribe en una linea

Una IA rara vez escribe un `for` con `append`. Casi siempre usa comprehensions. Es el mismo patron pero compacto.

**Antes de ejecutar:** traduce mentalmente la comprehension a un for-loop normal.

In [ ]:
orders = [
    {"symbol": "BTCUSDT", "side": "buy",  "price": 100000, "size": 0.10},
    {"symbol": "BTCUSDT", "side": "sell", "price": 100020, "size": 0.08},
    {"symbol": "ETHUSDT", "side": "buy",  "price": 3520,   "size": 1.40},
    {"symbol": "ETHUSDT", "side": "sell", "price": 3530,   "size": 0.50},
    {"symbol": "BTCUSDT", "side": "buy",  "price": 99980,  "size": 0.05},
]

# --- Estilo L1/L2: for + append ---
notionals_v1 = []
for o in orders:
    notionals_v1.append(o["price"] * o["size"])

# --- Estilo IA: list comprehension ---
notionals_v2 = [o["price"] * o["size"] for o in orders]

print("for+append: ", notionals_v1)
print("comprehension:", notionals_v2)
print("iguales?", notionals_v1 == notionals_v2)

# --- Con filtro ---
# Solo buys de BTC
btc_buys_v1 = []
for o in orders:
    if o["side"] == "buy" and o["symbol"] == "BTCUSDT":
        btc_buys_v1.append(o)

btc_buys_v2 = [o for o in orders if o["side"] == "buy" and o["symbol"] == "BTCUSDT"]

print("\nBTC buys:", len(btc_buys_v2))
for b in btc_buys_v2:
    print(f"  {b['size']} @ {b['price']}")

# --- Dict comprehension (bonus) ---
notional_by_symbol = {
    o["symbol"]: o["price"] * o["size"]
    for o in orders
}
# Cuidado: si hay duplicados, el ultimo gana
print("\nnotional_by_symbol:", notional_by_symbol)

## 5. Decoradores — el patron detras de `@property`

`@property` es un decorador. Un decorador es una funcion que envuelve otra funcion para anadirle comportamiento. La IA los usa constantemente.

**Antes de ejecutar:** ¿que crees que imprimira `log_call` antes de cada llamada?

In [ ]:
# Un decorador envuelve una funcion en otra funcion
def log_call(func):
    """Decorador que imprime cuando se llama una funcion."""
    def wrapper(*args, **kwargs):
        print(f"  [LOG] llamando a {func.__name__}()")
        result = func(*args, **kwargs)
        print(f"  [LOG] {func.__name__}() devolvio {result}")
        return result
    return wrapper


# Sin decorador
def notional_plain(price, size):
    return price * size

# Con decorador — el @ es azucar sintactico
@log_call
def notional_logged(price, size):
    return price * size

print("--- sin decorador ---")
print(notional_plain(100000, 0.10))

print("\n--- con decorador ---")
print(notional_logged(100000, 0.10))

# Lo que @log_call hace internamente:
# notional_logged = log_call(notional_logged)
# Es decir, reemplaza la funcion por la version envuelta.

print("\n--- @property es un decorador built-in ---")
print("tracker.cash llama internamente a un metodo,")
print("pero lo usas como si fuera un atributo:", tracker.cash)

## 6. Evaluar codigo completo generado por IA

Ahora juntamos todo. Imagina que le pides a un LLM: *"Genera un Order mejorado con type hints, @property para notional, y un metodo validate que lance ValueError si el precio es negativo."*

**Antes de ejecutar:** lee el codigo y busca si hay algun error sutil.

In [ ]:
# --- Codigo "generado por IA" (simulado) ---
class EnhancedOrder:
    """An improved Order with validation and computed properties."""

    VALID_SIDES = ("buy", "sell")

    def __init__(self, symbol: str, side: str, price: float, size: float) -> None:
        self._symbol = symbol
        self._side = side
        self._price = price
        self._size = size
        self.validate()

    def validate(self) -> None:
        """Raise ValueError if order parameters are invalid."""
        if self._price <= 0:
            raise ValueError(f"price must be positive, got {self._price}")
        if self._size <= 0:
            raise ValueError(f"size must be positive, got {self._size}")
        if self._side not in self.VALID_SIDES:
            raise ValueError(f"side must be one of {self.VALID_SIDES}, got {self._side}")

    @property
    def symbol(self) -> str:
        return self._symbol

    @property
    def side(self) -> str:
        return self._side

    @property
    def price(self) -> float:
        return self._price

    @property
    def size(self) -> float:
        return self._size

    @property
    def notional(self) -> float:
        """Computed property — no parentheses needed."""
        return self._price * self._size

    def describe(self) -> str:
        return f"{self._side} {self._size:.2f} {self._symbol} @ {self._price:,.0f}"

    def __repr__(self) -> str:
        return f"EnhancedOrder({self._symbol}, {self._side}, {self._price}, {self._size})"


# Probamos el caso feliz
order = EnhancedOrder("BTCUSDT", "buy", 100000, 0.10)
print(order.describe())
print("notional:", order.notional)  # sin parentesis — es @property
print(repr(order))

# Probamos la validacion
print("\n--- Validacion ---")
for bad_params in [
    ("BTCUSDT", "buy", -100, 0.10),   # precio negativo
    ("BTCUSDT", "buy", 100000, 0),     # size cero
    ("BTCUSDT", "hold", 100000, 0.10), # side invalido
]:
    try:
        bad_order = EnhancedOrder(*bad_params)
    except ValueError as e:
        print(f"  REJECTED: {e}")

## 7. Mini sistema completo: de prompt a codigo evaluado

Combinemos todo: `Trade`, `PositionTracker` con `@property`, comprehensions, y `try/except`. Esto es lo que un LLM podria generar — y ahora puedes leerlo.

**Antes de ejecutar:** sigue la historia trade a trade y predice el equity final.

In [ ]:
# Mini sistema usando todos los patrones de esta leccion

# 1. Datos
trade_log = [
    {"symbol": "BTCUSDT", "side": "buy",  "price": 100000, "size": 0.10},
    {"symbol": "BTCUSDT", "side": "buy",  "price": 99950,  "size": 0.05},
    {"symbol": "BTCUSDT", "side": "sell", "price": 100200, "size": 0.08},
    {"symbol": "ETHUSDT", "side": "buy",  "price": 3500,   "size": 2.00},  # otro activo — lo ignoramos
]

# 2. Filtrar con comprehension
btc_trades = [t for t in trade_log if t["symbol"] == "BTCUSDT"]
print(f"trades BTC: {len(btc_trades)} de {len(trade_log)} totales")

# 3. Procesar con try/except
tracker = PositionTracker()

for t_data in btc_trades:
    try:
        trade = Trade(t_data["symbol"], t_data["side"], t_data["price"], t_data["size"])
        tracker.apply_trade(trade)
        print(f"  applied: {trade.side} {trade.size} @ {trade.price} → cash={tracker.cash:.2f}")
    except Exception as e:
        print(f"  ERROR: {e}")

# 4. Resultado
mark = 100150
print(f"\n--- Resultado ---")
print(f"cash:     {tracker.cash:,.2f}")      # @property
print(f"position: {tracker.position:.4f}")    # @property
print(f"equity:   {tracker.equity(mark):,.2f} (mark={mark:,})")
print(f"\nNotional de ordenes originales:")
notionals = [t["price"] * t["size"] for t in btc_trades]  # comprehension
for t, n in zip(btc_trades, notionals):
    print(f"  {t['side']} → {n:,.2f}")

## 8. Tu turno

Crea una clase `TradeAnalyzer` que reciba una lista de diccionarios de trades y ofrezca:

- `@property total_notional` → suma de `price * size` de todos los trades
- `@property buy_count` → numero de trades con `side == "buy"` (usa comprehension)
- un metodo `summary() -> str` que devuelva un string descriptivo

Usa type hints en todos los metodos. Envuelve el calculo en `try/except` por si la lista esta vacia.

In [ ]:
class TradeAnalyzer:
    def __init__(self, trades: list[dict]) -> None:
        self._trades = trades

    # TODO: @property total_notional
    # TODO: @property buy_count
    # TODO: def summary(self) -> str

    pass


# Test
sample_trades = [
    {"symbol": "BTCUSDT", "side": "buy",  "price": 100000, "size": 0.10},
    {"symbol": "BTCUSDT", "side": "sell", "price": 100020, "size": 0.08},
    {"symbol": "BTCUSDT", "side": "buy",  "price": 99980,  "size": 0.05},
]

analyzer = TradeAnalyzer(sample_trades)
print("TODO -> implementa total_notional, buy_count, summary")

## 9. Una posible solucion

Comparala con tu enfoque. Fijate en como se combinan `@property`, comprehensions, type hints y `try/except` en una sola clase.

In [ ]:
class TradeAnalyzer:
    def __init__(self, trades: list[dict]) -> None:
        self._trades = trades

    @property
    def total_notional(self) -> float:
        try:
            return sum(t["price"] * t["size"] for t in self._trades)
        except (KeyError, TypeError):
            return 0.0

    @property
    def buy_count(self) -> int:
        return len([t for t in self._trades if t["side"] == "buy"])

    def summary(self) -> str:
        n = len(self._trades)
        return (
            f"{n} trades | "
            f"{self.buy_count} buys | "
            f"total notional: {self.total_notional:,.2f}"
        )


# Test con datos
analyzer = TradeAnalyzer(sample_trades)
print("total_notional:", analyzer.total_notional)
print("buy_count:", analyzer.buy_count)
print("summary:", analyzer.summary())

# Test con lista vacia
empty = TradeAnalyzer([])
print("\nempty notional:", empty.total_notional)
print("empty summary:", empty.summary())

## Cierre

Que deberia haberte quedado claro:

- **try/except:** captura errores sin crashear. Esencial cuando evaluas codigo generado por IA.
- **Type hints:** documentacion en la firma. `price: float` te dice que espera sin leer el cuerpo.
- **@property:** acceso controlado. `tracker.cash` sin parentesis, pero es un metodo por debajo.
- **Comprehensions:** `[x for x in items if cond]` reemplaza for+append. La IA los usa siempre.
- **Decoradores:** `@algo` envuelve una funcion. `@property` es uno de ellos.

**Siguiente paso:** ya sabes Python, OOP y como trabajar con IA. En la siguiente clase toca datos reales: microestructura de mercado con BTC.